# Kimee: final models on all 707 files (A100) — Phase 7 prep B

Phase 6 kept the recipe unchanged, so the only new thing here is **more data**: the 141 val files join training
(`--train-all`, 1,154 annotator samples instead of 921). Same recipe as exp012 / exp007 (`--label-policy all`,
flips + brightness/contrast, seed 42), 7,500 iterations instead of 6,000 (same number of passes over the data).
Then flip TTA (none / h / v / hv) on the test set, as in exp101.

**Val can NOT score these models** (they have seen val). The val files are still written, only as a sanity check
(they should score far above 0.45). The only check is the public LB vs exp101 (0.39).

| Step | Name | What | Time on A100 (estimate) |
|---|---|---|---|
| 0 | probe | 20 training steps with `--train-all` | ~5 min |
| 1 | `x101_2048_m28_all` | X101, 56x56 mask head, 2048, all files; then 4 views | ~4 h + ~10 min |
| 2 | `r50_2048_all` | R50, 2048, all files; then 4 views | ~2.5–3 h + ~10 min |

**Runtime:** A100 GPU + High-RAM, ~8 units/h, ~50–55 units in total. Close other Colab sessions; keep this tab open.
**If Colab disconnects:** run sections 1–4 again, then the cell of the interrupted step: training resumes from its
last checkpoint on Drive (every 1,000 iterations); finished trainings and prediction files are skipped.
Checkpoints: `solar_filament/output/detectron2/<name>/` (new folders; existing models are not touched).
Prediction files: `solar_filament/Phase 7/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/solar_filament"
OUT_DIR = f"{DRIVE_DIR}/output"            # checkpoints (persistent)
FINAL_DIR = f"{DRIVE_DIR}/Phase 7"         # raw predictions to send back
os.makedirs(FINAL_DIR, exist_ok=True)
os.environ["SOLAR_OUTPUT_DIR"] = OUT_DIR
print("Drive ready:", OUT_DIR)

## 2. Get the latest code (pulls if the repo is already cloned)

In [ ]:
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

## 3. Install dependencies (Detectron2 build ~5-8 min)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
print("torch", torch.__version__, "| detectron2", detectron2.__version__, "| GPU", torch.cuda.get_device_name(0))

## 4. Quick checks (~1 min) and helpers

In [ ]:
import shutil, subprocess, time

def sh(cmd):
    """Run one command (output shown live); stop the notebook if it fails."""
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")
    print(f"--- done in {(time.time() - t0) / 60:.1f} min")

ITERS = 7500   # 6,000 x 1,154 / 921 samples: same passes over the data as the 566-file models

def train_all(name, flags):
    """Train on all 707 files; skipped if the model is already finished (resumes otherwise)."""
    if os.path.exists(f"{OUT_DIR}/detectron2/{name}/model_final.pth"):
        print(name, "already trained, skipped.")
        return
    sh(f"python run_pipeline.py --step train --name {name} {flags} --label-policy all --train-all --max-iter {ITERS}")

def dump_views(name, flags):
    """Predict val + test on the original and the 3 flipped images; copy files to FINAL_DIR; skip finished ones."""
    weights = f"{OUT_DIR}/detectron2/{name}/model_final.pth"
    for flip in ("", "h", "v", "hv"):
        tag = f"_flip{flip}" if flip else ""
        if all(os.path.exists(f"{FINAL_DIR}/raw_preds_{name}{tag}_{s}.json.gz") for s in ("val", "test")):
            print(f"{name}{tag}: already done, skipped.")
            continue
        sh(f"python run_pipeline.py --step dump --name {name} {flags} --weights '{weights}'"
           + (f" --flip {flip}" if flip else ""))
        for s in ("val", "test"):
            shutil.copy(f"{OUT_DIR}/raw_preds_{name}{tag}_{s}.json.gz", FINAL_DIR)
    print(name, "files in", FINAL_DIR, ":", sorted(f for f in os.listdir(FINAL_DIR) if name in f))

sh("python -m pytest -q -p no:warnings tests/test_pipeline_flags.py tests/test_flip.py tests/test_official_metric.py")

## Step 0 — Probe (~5 min)
Must pass, or the notebook stops. The log must say `filament_train` has **707 images**.

In [ ]:
FLAGS_M28 = "--backbone x101 --resolution 2048 --mask-res 28"
FLAGS_R50 = "--backbone r50 --resolution 2048"
sh(f"python run_pipeline.py --step train --name probe_all {FLAGS_M28} --label-policy all --train-all --max-iter 20 --no-resume")
shutil.rmtree(f"{OUT_DIR}/detectron2/probe_all", ignore_errors=True)
print("Probe passed. Check above: '[Dataset] Registered 'filament_train' (707 images)'")

## Step 1 — `x101_2048_m28_all` (~4 h)

In [ ]:
NAME_M28 = "x101_2048_m28_all"
train_all(NAME_M28, FLAGS_M28)
dump_views(NAME_M28, FLAGS_M28)

## Step 2 — `r50_2048_all` (~3 h)

In [ ]:
NAME_R50 = "r50_2048_all"
train_all(NAME_R50, FLAGS_R50)
dump_views(NAME_R50, FLAGS_R50)

## 5. Send results back
Download the 16 files (also in `MyDrive/solar_filament/Phase 7/`) into the laptop folder `results/phase7/`, then
tell Claude. Also copy, per model: the `Peak GPU memory` line and the last `time ... s/it` line.
Then **disconnect the runtime** (Runtime -> Disconnect and delete runtime) so it stops using units.

In [ ]:
from google.colab import files
for name in ("x101_2048_m28_all", "r50_2048_all"):
    for tag in ("", "_fliph", "_flipv", "_fliphv"):
        for s in ("val", "test"):
            f = f"{FINAL_DIR}/raw_preds_{name}{tag}_{s}.json.gz"
            if os.path.exists(f):
                files.download(f)
            else:
                print("not found (not finished?):", f)